# einops-rearrange composite — cx25: wrap ndarray with from_numpy, then rearrange axes

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-rearrange`, `tensor-wraps-ndarray`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-rearrange"
DD_ATOM_IDS = ["einops-rearrange", "tensor-wraps-ndarray"]
DD_SUBTOPICS = ["Einops: Rearrange", "PyTorch: tensor from ndarray"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Data pipelines almost always start in NumPy land — images decoded as `np.ndarray`, sensor logs loaded with `np.load`, etc. To feed them through a torch model we wrap with `t.from_numpy(arr)` (zero-copy view) and then reshape with `einops.rearrange` into the layout the model expects.

The composition exercises BOTH atoms: the wrap is load-bearing (we assert storage aliasing — no defensive copy), and the rearrange pattern is load-bearing (we assert the exact axis order). Together they form the canonical NumPy → torch → CNN-shape bridge.

### Composite Exercise — wrap ndarray with from_numpy, then rearrange axes

**Atoms exercised together**: `einops-rearrange`, `tensor-wraps-ndarray`

Implement `cx25_ndarray_to_nchw(arr)` that takes an `np.ndarray` of shape `(H, W, C)` (NumPy/HWC convention — what cv2 / PIL hand you) and returns a torch tensor of shape `(1, C, H, W)` (torch/NCHW convention).

1. **Wrap** — use `t.from_numpy(arr)` so the tensor shares storage with `arr` (no defensive copy).
2. **Rearrange** — use `einops.rearrange(..., 'h w c -> 1 c h w')` to insert the batch axis AND reorder HWC → CHW in one step.

The test asserts both the wrap (data_ptr aliasing) and the rearrange (exact byte-order match with `t.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)`).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx25_ndarray_to_nchw(arr):
    raise NotImplementedError

def _test_cx25():
    # Case A: distinguishable values — verify the axis order is correct.
    arr = np.arange(24, dtype=np.float32).reshape(2, 3, 4)  # (H=2, W=3, C=4)
    out = cx25_ndarray_to_nchw(arr)
    assert isinstance(out, t.Tensor), f'expected torch.Tensor, got {type(out)}'
    assert tuple(out.shape) == (1, 4, 2, 3), f'expected (1,4,2,3), got {tuple(out.shape)}'
    expected = t.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    assert t.equal(out, expected), 'axis order wrong — did you use h w c -> 1 c h w?'

    # Case B: aliasing — from_numpy must share storage; mutating arr must change the tensor.
    arr2 = np.zeros((3, 4, 2), dtype=np.float32)
    out2 = cx25_ndarray_to_nchw(arr2)
    arr2[0, 0, 0] = 99.0
    # After rearrange the tensor is a view; the (0, 0, 0, 0) slot in NCHW maps to (0, 0, 0) HWC.
    assert out2[0, 0, 0, 0].item() == 99.0, (
        'tensor must share storage with arr (from_numpy zero-copy). '
        'Did you call .clone() or t.tensor(arr)?'
    )

    # Case C: realistic CNN-ish image shape.
    img = np.random.RandomState(0).rand(32, 32, 3).astype(np.float32)
    out3 = cx25_ndarray_to_nchw(img)
    assert tuple(out3.shape) == (1, 3, 32, 32)
    assert out3.dtype == t.float32
    # Pluggable into a Conv2d.
    conv = t.nn.Conv2d(3, 8, kernel_size=3, padding=1)
    feat = conv(out3)
    assert feat.shape == (1, 8, 32, 32)
    _dd_passed.add('cx25')

_test_cx25()

<details><summary>Show solution — cx25</summary>

```python
def cx25_ndarray_to_nchw(arr):
    # Atom A: wrap the ndarray zero-copy.
    wrapped = t.from_numpy(arr)
    # Atom B: rearrange HWC -> NCHW in one named pattern (insert batch + reorder).
    return rearrange(wrapped, 'h w c -> 1 c h w')
```

`from_numpy` is the zero-copy half — the resulting tensor's storage IS the ndarray's buffer. `rearrange('h w c -> 1 c h w')` then constructs a non-contiguous view that permutes axes and inserts the batch dim — still sharing storage with `arr`. If you replace `from_numpy` with `t.tensor(arr)` the storage check fails; if you flip the rearrange order to `'h w c -> 1 h w c'` the byte-order check fails.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx25'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx25',
        'subtopics': ["Einops: Rearrange", "PyTorch: tensor from ndarray"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()